## 01 Load Data

This notebook downloads and reads the AWV cycling traffic data. It loads the site metadata, direction metadata, and monthly count files from May 2022 to April 2026. Additional datasets (holidays data, fuel prices data, weather data and major events data) used in the analysis are also loaded and read here.

### Importing packages

In [1]:
import pandas as pd
import requests
import time
from pathlib import Path

### Defining path

In [2]:
project_folder = Path("..")

raw_folder = project_folder / "data" / "raw"
processed_folder = project_folder / "data" / "processed"
external_folder = project_folder / "data" / "external"

raw_folder.mkdir(parents=True, exist_ok=True)
processed_folder.mkdir(parents=True, exist_ok=True)
external_folder.mkdir(parents=True, exist_ok=True)


## Loading AWV data

Defining the the AWV website

In [3]:
base_url = "https://opendata.apps.mow.vlaanderen.be/fietstellingen"

Required files

In [4]:
metadata_files = [
    "sites.csv",
    "richtingen.csv",
]

count_files = [
    "data-2022-05.csv",
    "data-2022-06.csv",
    "data-2022-07.csv",
    "data-2022-08.csv",
    "data-2022-09.csv",
    "data-2022-10.csv",
    "data-2022-11.csv",
    "data-2022-12.csv",

    "data-2023-01.csv",
    "data-2023-02.csv",
    "data-2023-03.csv",
    "data-2023-04.csv",
    "data-2023-05.csv",
    "data-2023-06.csv",
    "data-2023-07.csv",
    "data-2023-08.csv",
    "data-2023-09.csv",
    "data-2023-10.csv",
    "data-2023-11.csv",
    "data-2023-12.csv",

    "data-2024-01.csv",
    "data-2024-02.csv",
    "data-2024-03.csv",
    "data-2024-04.csv",
    "data-2024-05.csv",
    "data-2024-06.csv",
    "data-2024-07.csv",
    "data-2024-08.csv",
    "data-2024-09.csv",
    "data-2024-10.csv",
    "data-2024-11.csv",
    "data-2024-12.csv",

    "data-2025-01.csv",
    "data-2025-02.csv",
    "data-2025-03.csv",
    "data-2025-04.csv",
    "data-2025-05.csv",
    "data-2025-06.csv",
    "data-2025-07.csv",
    "data-2025-08.csv",
    "data-2025-09.csv",
    "data-2025-10.csv",
    "data-2025-11.csv",
    "data-2025-12.csv",

    "data-2026-01.csv",
    "data-2026-02.csv",
    "data-2026-03.csv",
    "data-2026-04.csv",
]

all_files = metadata_files + count_files


Downloading the files

In [5]:
for file_name in all_files:
    url = base_url + "/" + file_name
    save_path = raw_folder / file_name

    if save_path.exists():
        print("Already downloaded:", file_name)
    else:
        print("Downloading:", file_name)

        response = requests.get(url)
        response.raise_for_status()

        save_path.write_bytes(response.content)

        print("Saved:", save_path)

Already downloaded: sites.csv
Already downloaded: richtingen.csv
Already downloaded: data-2022-05.csv
Already downloaded: data-2022-06.csv
Already downloaded: data-2022-07.csv
Already downloaded: data-2022-08.csv
Already downloaded: data-2022-09.csv
Already downloaded: data-2022-10.csv
Already downloaded: data-2022-11.csv
Already downloaded: data-2022-12.csv
Already downloaded: data-2023-01.csv
Already downloaded: data-2023-02.csv
Already downloaded: data-2023-03.csv
Already downloaded: data-2023-04.csv
Already downloaded: data-2023-05.csv
Already downloaded: data-2023-06.csv
Already downloaded: data-2023-07.csv
Already downloaded: data-2023-08.csv
Already downloaded: data-2023-09.csv
Already downloaded: data-2023-10.csv
Already downloaded: data-2023-11.csv
Already downloaded: data-2023-12.csv
Already downloaded: data-2024-01.csv
Already downloaded: data-2024-02.csv
Already downloaded: data-2024-03.csv
Already downloaded: data-2024-04.csv
Already downloaded: data-2024-05.csv
Already do

Defining column names

In [6]:
site_columns = [
    "site_id",
    "site_number",
    "longitude",
    "latitude",
    "site_name",
    "domain",
    "road_number",
    "district",
    "municipality",
    "counting_interval_minutes",
    "installation_date",
]

direction_columns = [
    "site_id",
    "direction",
    "direction_description",
]

count_columns = [
    "site_id",
    "direction",
    "traffic_type",
    "start_time",
    "end_time",
    "count",
]

### Reading metadata files

In [7]:
sites = pd.read_csv(
    raw_folder / "sites.csv",
    header=None,
    names=site_columns,
)

directions = pd.read_csv(
    raw_folder / "richtingen.csv",
    header=None,
    names=direction_columns,
)

In [8]:
sites.head()

,site_id,site_number,longitude,latitude,site_name,domain,road_number,district,municipality,counting_interval_minutes,installation_date
0,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22
1,2,100052862,4.471690,51.275120,Brasschaat 2,Vlaamse Overheid A. Wegen enVerkeer,N0010002,AWV123,Brasschaat,15,2019-08-22
2,3,100052863,4.472220,51.275030,Brasschaat 1,Vlaamse Overheid A. Wegen enVerkeer,N0010001,AWV123,Brasschaat,15,2019-08-22
3,4,100052864,5.190110,51.160230,Balen 1,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22
4,5,100052865,5.190030,51.160180,Balen 2,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22


In [9]:
sites.info()

<class 'pandas.DataFrame'>
RangeIndex: 151 entries, 0 to 150
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   site_id                    151 non-null    int64  
 1   site_number                151 non-null    int64  
 2   longitude                  151 non-null    float64
 3   latitude                   151 non-null    float64
 4   site_name                  151 non-null    str    
 5   domain                     151 non-null    str    
 6   road_number                148 non-null    str    
 7   district                   148 non-null    str    
 8   municipality               150 non-null    str    
 9   counting_interval_minutes  151 non-null    int64  
 10  installation_date          151 non-null    str    
dtypes: float64(2), int64(3), str(6)
memory usage: 24.9 KB


In [10]:
directions.head()

,site_id,direction,direction_description
0,1,IN,Machelen Cyclists rich. Brucargo
1,1,OUT,Machelen Cyclists richting Machelen
2,2,IN,Brasschaat 2 Fietsers rich Merksem
3,2,OUT,Brasschaat 2 Fietsers rich Brasschaat
4,3,IN,Brasschaat 1 Fietsers rich Merksem


In [11]:
directions.info()

<class 'pandas.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   site_id                305 non-null    int64
 1   direction              305 non-null    str  
 2   direction_description  305 non-null    str  
dtypes: int64(1), str(2)
memory usage: 17.8 KB


### Reading monthly data

In [12]:
monthly_counts = {}

for file_name in count_files:
    print("Reading:", file_name)

    monthly_counts[file_name] = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

Reading: data-2022-05.csv
Reading: data-2022-06.csv
Reading: data-2022-07.csv
Reading: data-2022-08.csv
Reading: data-2022-09.csv
Reading: data-2022-10.csv
Reading: data-2022-11.csv
Reading: data-2022-12.csv
Reading: data-2023-01.csv
Reading: data-2023-02.csv
Reading: data-2023-03.csv
Reading: data-2023-04.csv
Reading: data-2023-05.csv
Reading: data-2023-06.csv
Reading: data-2023-07.csv
Reading: data-2023-08.csv
Reading: data-2023-09.csv
Reading: data-2023-10.csv
Reading: data-2023-11.csv
Reading: data-2023-12.csv
Reading: data-2024-01.csv
Reading: data-2024-02.csv
Reading: data-2024-03.csv
Reading: data-2024-04.csv
Reading: data-2024-05.csv
Reading: data-2024-06.csv
Reading: data-2024-07.csv
Reading: data-2024-08.csv
Reading: data-2024-09.csv
Reading: data-2024-10.csv
Reading: data-2024-11.csv
Reading: data-2024-12.csv
Reading: data-2025-01.csv
Reading: data-2025-02.csv
Reading: data-2025-03.csv
Reading: data-2025-04.csv
Reading: data-2025-05.csv
Reading: data-2025-06.csv
Reading: dat

In [13]:
len(monthly_counts)

48

In [14]:
monthly_counts["data-2022-05.csv"].head()

,site_id,direction,traffic_type,start_time,end_time,count
0,1,IN,FIETSERS,2022-05-01 00:00:00.0,2022-05-01 00:15:00.0,3.0
1,1,IN,FIETSERS,2022-05-01 00:15:00.0,2022-05-01 00:30:00.0,0.0
2,1,IN,FIETSERS,2022-05-01 00:30:00.0,2022-05-01 00:45:00.0,0.0
3,1,IN,FIETSERS,2022-05-01 00:45:00.0,2022-05-01 01:00:00.0,7.0
4,1,IN,FIETSERS,2022-05-01 01:00:00.0,2022-05-01 01:15:00.0,2.0


In [15]:
monthly_counts["data-2022-05.csv"].info()

<class 'pandas.DataFrame'>
RangeIndex: 368518 entries, 0 to 368517
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   site_id       368518 non-null  int64  
 1   direction     368518 non-null  str    
 2   traffic_type  368518 non-null  str    
 3   start_time    368518 non-null  str    
 4   end_time      368518 non-null  str    
 5   count         278376 non-null  float64
dtypes: float64(1), int64(1), str(4)
memory usage: 35.3 MB


In [16]:
monthly_counts["data-2022-05.csv"].shape

(368518, 6)

In [17]:
month_sizes = []

for file_name in count_files:
    number_of_rows = monthly_counts[file_name].shape[0]
    number_of_columns = monthly_counts[file_name].shape[1]

    month_sizes.append({
        "file_name": file_name,
        "rows": number_of_rows,
        "columns": number_of_columns,
    })

month_sizes = pd.DataFrame(month_sizes)

month_sizes

,file_name,rows,columns
0,data-2022-05.csv,368518,6
1,data-2022-06.csv,486830,6
2,data-2022-07.csv,632122,6
3,data-2022-08.csv,668966,6
4,data-2022-09.csv,714104,6
5,data-2022-10.csv,767808,6
6,data-2022-11.csv,740358,6
7,data-2022-12.csv,763118,6
8,data-2023-01.csv,780396,6
9,data-2023-02.csv,720384,6


In [18]:
missing_count_summary = []

for file_name, df in monthly_counts.items():
    missing_count_summary.append({
        "file_name": file_name,
        "rows": df.shape[0],
        "missing_counts": df["count"].isna().sum(),
        "missing_start_time": df["start_time"].isna().sum(),
        "missing_end_time": df["end_time"].isna().sum(),
    })

missing_count_summary = pd.DataFrame(missing_count_summary)
missing_count_summary

,file_name,rows,missing_counts,missing_start_time,missing_end_time
0,data-2022-05.csv,368518,90142,0,0
1,data-2022-06.csv,486830,124142,0,0
2,data-2022-07.csv,632122,117048,0,0
3,data-2022-08.csv,668966,34446,0,0
4,data-2022-09.csv,714104,65592,0,0
5,data-2022-10.csv,767808,29656,0,0
6,data-2022-11.csv,740358,6,0,0
7,data-2022-12.csv,763118,1494,0,0
8,data-2023-01.csv,780396,17086,0,0
9,data-2023-02.csv,720384,15790,0,0


In [19]:
missing_count_summary["missing_count_percent"] = (
    missing_count_summary["missing_counts"] / missing_count_summary["rows"] * 100
).round(2)

missing_count_summary

,file_name,rows,missing_counts,missing_start_time,missing_end_time,missing_count_percent
0,data-2022-05.csv,368518,90142,0,0,24.46
1,data-2022-06.csv,486830,124142,0,0,25.50
2,data-2022-07.csv,632122,117048,0,0,18.52
3,data-2022-08.csv,668966,34446,0,0,5.15
4,data-2022-09.csv,714104,65592,0,0,9.19
5,data-2022-10.csv,767808,29656,0,0,3.86
6,data-2022-11.csv,740358,6,0,0,0.00
7,data-2022-12.csv,763118,1494,0,0,0.20
8,data-2023-01.csv,780396,17086,0,0,2.19
9,data-2023-02.csv,720384,15790,0,0,2.19


Most monthly count files contain missing values in the `count` column. The timestamp columns do not contain missing values. Missing counts are especially common in the first months of the study period, particularly from April to July 2022. 

## Loading external data

### Loading public holidays data

In [20]:
years = range(2022, 2027)

public_holidays = []

for year in years:
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/BE"
    df = pd.read_json(url)
    public_holidays.append(df)

public_holidays = pd.concat(public_holidays, ignore_index=True)

public_holidays["date"] = pd.to_datetime(public_holidays["date"])

public_holidays = public_holidays[
    (public_holidays["date"] >= "2022-05-01") &
    (public_holidays["date"] <= "2026-04-30")
]

public_holidays = public_holidays.rename(columns={
    "name": "holiday_name"
})

public_holidays = public_holidays[
    ["date", "holiday_name"]
].copy()

public_holidays

,date,holiday_name
4,2022-05-01,Labour Day
5,2022-05-26,Ascension Day
6,2022-05-27,Day after Ascension Day
7,2022-06-06,Whit Monday
8,2022-07-21,Belgian National Day
9,2022-08-15,Assumption Day
10,2022-11-01,All Saints' Day
11,2022-11-11,Armistice Day
12,2022-12-25,Christmas Day
13,2022-12-26,St. Stephen's Day


Saving public holidays data

In [21]:
public_holidays.to_csv(
    external_folder / "public_holidays.csv",
    index=False
)

### Loading school holidays data

In [22]:
school_holidays_url = "https://openholidaysapi.org/SchoolHolidays"

date_ranges = [
    ("2022-05-01", "2025-03-31"),
    ("2025-04-01", "2026-04-30"),
]

school_holidays = []

for valid_from, valid_to in date_ranges:
    params = {
        "countryIsoCode": "BE",
        "validFrom": valid_from,
        "validTo": valid_to,
        "languageIsoCode": "EN",
    }

    response = requests.get(school_holidays_url, params=params)
    response.raise_for_status()

    data = response.json()
    temp = pd.json_normalize(data)
    school_holidays.append(temp)

school_holidays = pd.concat(school_holidays, ignore_index=True)

school_holidays

,id,startDate,endDate,type,name,regionalScope,temporalScope,nationwide,groups
0,47817b89-3767-46bc-8f78-6e24119742e6,2022-07-01,2022-08-31,School,"[{'language': 'EN', 'text': 'Summer Holidays'}]",Regional,FullDay,False,"[{'code': 'BE-DE', 'shortName': 'DE'}]"
1,d987f457-2367-4ef0-8918-64828e596634,2022-07-01,2022-08-31,School,"[{'language': 'EN', 'text': 'Summer Holidays'}]",Regional,FullDay,False,"[{'code': 'BE-NL', 'shortName': 'NL'}]"
2,f9a31448-a3f8-49d3-b46b-db200dc98f7a,2022-07-01,2022-08-28,School,"[{'language': 'EN', 'text': 'Summer Holidays'}]",Regional,FullDay,False,"[{'code': 'BE-FR', 'shortName': 'FR'}]"
3,07b51fac-63a4-4849-bbec-59ca8f1c49a3,2022-09-27,2022-09-27,School,"[{'language': 'EN', 'text': 'Day of the French...",Regional,FullDay,False,"[{'code': 'BE-FR', 'shortName': 'FR'}]"
4,7a6be322-2c1c-4827-b69f-2ffb3098aeba,2022-10-24,2022-11-04,School,"[{'language': 'EN', 'text': 'All Saints Holida...",Regional,FullDay,False,"[{'code': 'BE-FR', 'shortName': 'FR'}]"
...,...,...,...,...,...,...,...,...,...
63,b4a98912-e118-4090-877c-682086df4ebe,2026-02-16,2026-02-21,School,"[{'language': 'EN', 'text': 'Carnival Holidays'}]",Regional,FullDay,False,"[{'code': 'BE-DE', 'shortName': 'DE'}]"
64,fccb17bd-aeac-4211-9b72-8db4f69aa4c3,2026-02-16,2026-03-01,School,"[{'language': 'EN', 'text': 'Carnival Holidays'}]",Regional,FullDay,False,"[{'code': 'BE-FR', 'shortName': 'FR'}]"
65,3b35acc8-1d1f-4291-a07f-3f181852afd2,2026-04-06,2026-04-19,School,"[{'language': 'EN', 'text': 'Spring Holidays'}]",Regional,FullDay,False,"[{'code': 'BE-NL', 'shortName': 'NL'}]"
66,d76adc18-3e8f-4a6d-ac9e-2a9b0e4db5f8,2026-04-06,2026-04-18,School,"[{'language': 'EN', 'text': 'Spring Holidays'}]",Regional,FullDay,False,"[{'code': 'BE-DE', 'shortName': 'DE'}]"


School holidays in Flanders

In [23]:
def extract_group_code(groups):
    return groups[0]["code"]


def extract_holiday_name(name):
    return name[0]["text"]


school_holidays["group_code"] = school_holidays["groups"].apply(extract_group_code)
school_holidays["school_holiday_name"] = school_holidays["name"].apply(extract_holiday_name)

school_holidays_flanders = school_holidays[
    school_holidays["group_code"] == "BE-NL"
].copy()

school_holidays_flanders = school_holidays_flanders.rename(columns={
    "startDate": "start_date",
    "endDate": "end_date"
})

school_holidays_flanders["start_date"] = pd.to_datetime(school_holidays_flanders["start_date"])
school_holidays_flanders["end_date"] = pd.to_datetime(school_holidays_flanders["end_date"])

school_holidays_flanders = school_holidays_flanders[
    ["start_date", "end_date", "school_holiday_name"]
].copy()

school_holidays_flanders

,start_date,end_date,school_holiday_name
1,2022-07-01,2022-08-31,Summer Holidays
6,2022-10-31,2022-11-06,All Saints Holidays
8,2022-12-26,2023-01-08,Winter Holidays
13,2023-02-20,2023-02-26,Carnival Holidays
14,2023-04-03,2023-04-16,Spring Holidays
17,2023-07-01,2023-08-31,Summer Holidays
22,2023-10-30,2023-11-05,All Saints Holidays
25,2023-12-25,2024-01-07,Winter Holidays
29,2024-02-12,2024-02-18,Carnival Holidays
32,2024-04-01,2024-04-14,Spring Holidays


Saving school holidays data

In [24]:
school_holidays_flanders.to_csv(
    external_folder / "school_holidays_flanders.csv",
    index=False
)

### Loading fuel prices data

In [25]:
fuel_url = "https://energy.ec.europa.eu/document/download/906e60ca-8b6a-44e7-8589-652854d2fd3f_en?filename=Weekly_Oil_Bulletin_Prices_History_maticni_4web.xlsx"

fuel_raw = pd.read_excel(
    fuel_url,
    sheet_name="Prices with taxes",
    header=[0, 1, 2]
)

fuel_raw.head()

,Consumer prices of petroleum products inclusive of duties and taxes,CTR,EU_price_with_tax_euro95,EU_price_with_tax_diesel,EU_price_with_tax_heEUing_oil,EU_price_with_tax_fuel_oil_1,EU_price_with_tax_fuel_oil_2,EU_price_with_tax_LPG,CTR,EUR_price_with_tax_euro95,...,SK_price_with_tax_fuel_oil_2,SK_price_with_tax_LPG,CTR,UK_exchange_rate,UK_price_with_tax_euro95,UK_price_with_tax_diesel,UK_price_with_tax_heUKing_oil,UK_price_with_tax_fuel_oil_1,UK_price_with_tax_fuel_oil_2,UK_price_with_tax_LPG
,Unnamed: 0_level_1,Unnamed: 1_level_1,Euro-super 95 (I),Gas oil automobile Automotive gas oil Dieselkraftstoff (I),Gas oil de chauffage Heating gas oil Heizöl (II),Fuel oil - Schweres Heizöl (III) Soufre,Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%,GPL pour moteur LPG motor fuel,Unnamed: 8_level_1,Euro-super 95 (I),...,Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%,GPL pour moteur LPG motor fuel,Unnamed: 218_level_1,Unnamed: 219_level_1,Euro-super 95 (I),Gas oil automobile Automotive gas oil Dieselkraftstoff (I),Gas oil de chauffage Heating gas oil Heizöl (II),Fuel oil - Schweres Heizöl (III) Soufre,Fuel oil -Schweres Heizöl (III) Soufre > 1% Sulphur > 1% Schwefel > 1%,GPL pour moteur LPG motor fuel
,Date,Unnamed: 1_level_2,1000 l,1000 l,1000 l,t,t,1000 l,Unnamed: 8_level_2,1000 l,...,t,1000 l,Unnamed: 218_level_2,Unnamed: 219_level_2,1000 l,1000 l,1000 l,t,t,1000 l
0,2026-05-11 00:00:00,EU_,1871.179697,1930.543967,1415.169013,801.772763,656.054540,943.704034,EUR_,1930.366984,...,NaN,906.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-05-04 00:00:00,EU_,1861.113719,1975.34454,1480.893315,804.254047,631.715587,938.463537,EUR_,1909.034925,...,NaN,912.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-27 00:00:00,EU_,1841.515072,1974.343876,1421.360167,766.447376,595.076998,934.959553,EUR_,1896.497943,...,NaN,901.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-20 00:00:00,EU_,1827.449889,2007.925502,1463.983752,782.251889,588.093422,933.525010,EUR_,1887.468271,...,NaN,901.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-13 00:00:00,EU_,1853.723723,2099.719327,1544.892508,731.107881,631.196930,922.967746,EUR_,1915.431289,...,NaN,852.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
fuel_raw = pd.read_excel(
    fuel_url,
    sheet_name="Prices with taxes",
    header=[0, 1, 2]
)

date_column = fuel_raw.columns[
    fuel_raw.columns.get_level_values(2) == "Date"
][0]

petrol_column = fuel_raw.columns[
    (fuel_raw.columns.get_level_values(0) == "BE_price_with_tax_euro95") &
    (fuel_raw.columns.get_level_values(1) == "Euro-super 95  (I)") &
    (fuel_raw.columns.get_level_values(2) == "1000 l")
][0]

fuel_prices = fuel_raw[[date_column, petrol_column]].copy()

fuel_prices.columns = [
    "date",
    "petrol_95_eur_per_1000_l"
]

fuel_prices["date"] = pd.to_datetime(
    fuel_prices["date"],
    dayfirst=True,
    errors="coerce"
)

fuel_prices = fuel_prices[
    (fuel_prices["date"] >= "2022-05-01") &
    (fuel_prices["date"] <= "2026-04-30")
].copy()

fuel_prices = fuel_prices.sort_values("date").reset_index(drop=True)

fuel_prices

,date,petrol_95_eur_per_1000_l
0,2022-05-02,1821.53
1,2022-05-09,1875.58
2,2022-05-16,1925.81
3,2022-05-23,1967.77
4,2022-05-30,1947.38
...,...,...
204,2026-03-30,1848.31
205,2026-04-06,1896.50
206,2026-04-13,1874.94
207,2026-04-20,1834.52


Saving fuel price data

In [27]:
fuel_prices.to_csv(
    external_folder / "fuel_prices_belgium.csv",
    index=False
)

### Loading weather data

Weather data were not downloaded separately for every counting site. Some AWV sites are geographically very close to each other, so they are expected to have almost identical weather conditions.

To avoid unnecessary repeated API requests, the latitude and longitude of each site were rounded to two decimal places. Sites with the same rounded coordinates were treated as sharing the same weather location. Weather data were downloaded once for each unique rounded coordinate pair and then merged back to all sites with those rounded coordinates. This means that nearby sites may use the same weather observations, which is reasonable because weather conditions are unlikely to differ meaningfully over such short distances.

In [28]:
weather_sites = sites[
    ["site_id", "latitude", "longitude", "site_name", "municipality"]
].copy()

weather_sites["latitude"] = pd.to_numeric(
    weather_sites["latitude"],
    errors="coerce"
)

weather_sites["longitude"] = pd.to_numeric(
    weather_sites["longitude"],
    errors="coerce"
)

weather_sites = weather_sites.dropna(
    subset=["latitude", "longitude"]
).copy()

weather_sites["latitude_rounded"] = weather_sites["latitude"].round(2)
weather_sites["longitude_rounded"] = weather_sites["longitude"].round(2)

weather_locations = weather_sites[
    ["latitude_rounded", "longitude_rounded"]
].drop_duplicates().reset_index(drop=True)

weather_locations.shape

(98, 2)

In [29]:
start_date = "2025-05-01"
end_date = "2026-04-30"

In [30]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"

In [31]:
def download_weather_for_location(latitude, longitude, start_date, end_date):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,precipitation,rain,snowfall,wind_speed_10m",
        "timezone": "Europe/Brussels",
    }

    response = requests.get(weather_url, params=params)

    if response.status_code == 429:
        print("Too many requests. Waiting 60 seconds...")
        time.sleep(60)

        response = requests.get(weather_url, params=params)

    data = response.json()

    weather = pd.DataFrame(data["hourly"])

    weather["latitude_rounded"] = latitude
    weather["longitude_rounded"] = longitude

    return weather

In [32]:
all_weather_locations = []

for index, row in weather_locations.iterrows():
    latitude = row["latitude_rounded"]
    longitude = row["longitude_rounded"]

    print("Downloading weather for location:", latitude, longitude)

    location_weather = download_weather_for_location(
        latitude=latitude,
        longitude=longitude,
        start_date=start_date,
        end_date=end_date,
    )

    all_weather_locations.append(location_weather)

    time.sleep(3)

In [33]:
weather_by_location = pd.concat(
    all_weather_locations,
    ignore_index=True
)

weather_by_location["time"] = pd.to_datetime(
    weather_by_location["time"]
)

weather_by_location.head()

,time,temperature_2m,precipitation,rain,snowfall,wind_speed_10m,latitude_rounded,longitude_rounded
0,2025-05-01 00:00:00,15.5,0.0,0.0,0.0,6.0,50.92,4.46
1,2025-05-01 01:00:00,14.5,0.0,0.0,0.0,5.9,50.92,4.46
2,2025-05-01 02:00:00,13.6,0.0,0.0,0.0,6.1,50.92,4.46
3,2025-05-01 03:00:00,12.5,0.0,0.0,0.0,5.5,50.92,4.46
4,2025-05-01 04:00:00,11.6,0.0,0.0,0.0,5.3,50.92,4.46


In [34]:
weather_data = weather_by_location.merge(
    weather_sites[
        ["site_id", "latitude_rounded", "longitude_rounded"]
    ],
    on=["latitude_rounded", "longitude_rounded"],
    how="left",
)

weather_data.head()

,time,temperature_2m,precipitation,rain,snowfall,wind_speed_10m,latitude_rounded,longitude_rounded,site_id
0,2025-05-01 00:00:00,15.5,0.0,0.0,0.0,6.0,50.92,4.46,1
1,2025-05-01 01:00:00,14.5,0.0,0.0,0.0,5.9,50.92,4.46,1
2,2025-05-01 02:00:00,13.6,0.0,0.0,0.0,6.1,50.92,4.46,1
3,2025-05-01 03:00:00,12.5,0.0,0.0,0.0,5.5,50.92,4.46,1
4,2025-05-01 04:00:00,11.6,0.0,0.0,0.0,5.3,50.92,4.46,1


In [35]:
weather_data.to_csv(
    external_folder / "raw_weather_data.csv",
    index=False
)